Phase 1: NBA Spatial Data Extraction. This notebook connects to the live NBA REST API to extract raw shot coordinates for the Golden State Warriors. It performs initial ETL (Extract, Transform, Load) to clean the JSON response into a structured DataFrame.

## Data

In [ ]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster, shotchartdetail
import random
import numpy as np
import pandas as pd
from pathlib import Path
np.random.seed(42)
random.seed(42)
import time


print("Tools loaded successfully.")

In [ ]:
# Find GSW ID
nba_teams = teams.get_teams()
gsw_id = [team for team in nba_teams if team['abbreviation'] == 'GSW'][0]['id']

# Get the Roster
print("Fetching the 2021-22 Championship Roster...")
roster = commonteamroster.CommonTeamRoster(team_id=gsw_id, season='2021-22')
df_roster = roster.get_data_frames()[0]

# Create the checklist
players_to_fetch = df_roster[['PLAYER_ID', 'PLAYER']]
print(f"Target Acquired: {len(players_to_fetch)} players found on the roster.")

API calls to the NBA server frequently time out. This loop incorporates a try/except block with a 3-attempt retry limit and rate-limiting (time.sleep) to ensure the pipeline doesn't crash during server lag.

In [ ]:
all_shots_data = []

for index, row in players_to_fetch.iterrows():
    player_id = row['PLAYER_ID']
    player_name = row['PLAYER']

    print(f"Extracting spatial data for: {player_name}...")

    success = False
    attempts = 0

    while not success and attempts < 3:
        try:
            shot_data = shotchartdetail.ShotChartDetail(
                team_id=gsw_id,
                player_id=player_id,
                context_measure_simple='FGA',
                season_nullable='2021-22',
                season_type_all_star='Regular Season',
                timeout=60
            )

            df_player_shots = shot_data.get_data_frames()[0]
            all_shots_data.append(df_player_shots)
            success = True

        except Exception as e:
            attempts += 1
            print(f"  -> Server timed out (Attempt {attempts}/3). Pausing for 5 seconds...")
            time.sleep(5)

    time.sleep(1)

print("\nAll player data successfully extracted and stored in memory!")

The raw API response contains redundant metadata (game IDs, arena names). This step filters the dataset down strictly to the analytical features needed for spatial mapping and Expected Value calculations.

## Cleaning

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
raw_dir = repo_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

df_master = pd.concat(all_shots_data, ignore_index=True)

# Clean it
columns_to_keep = [
    'PLAYER_NAME', 'GAME_DATE', 'ACTION_TYPE', 'SHOT_TYPE',
    'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_DISTANCE',
    'LOC_X', 'LOC_Y', 'SHOT_MADE_FLAG'
]
df_clean = df_master[columns_to_keep]

# Export it
filename = raw_dir / "gsw_2022_roster_shots.csv"
df_clean.to_csv(filename, index=False)
print(f"Master dataset compiled: {len(df_clean)} total shots.")


## Results

In [ ]:
df_clean.head()
